# 004 RWKV: Testing and Experimenting


In [1]:
# Cell 1: Install required libraries using uv and the active kernel
import sys

# We use {sys.executable} to guarantee uv installs into THIS exact notebook environment
!uv pip install --python {sys.executable} transformers torch accelerate rwkv ninja

Using Python 3.14.7 environment at: /home/bpeng/code/grimoire/.venv
Resolved 57 packages in 792ms                                        
Prepared 40 packages in 6m 02s                                               nvidia-cudnn-cu13      ------------------------------ 527.20 MiB/527.48 MiB     torch                  ------------------------------ 528.58 MiB/528.93 MiB     torch                  ------------------------------ 403.00 MiB/528.93 MiB     torch                  ------------------------------ 237.41 MiB/528.93 MiB     torch                  ------------------------------ 204.86 MiB/528.93 MiB     torch                  ------------------------------ 202.20 MiB/528.93 MiB     torch                  ------------------------------ 192.33 MiB/528.93 MiB     torch                  ------------------------------ 164.08 MiB/528.93 MiB     torch                  ------------------------------ 139.06 MiB/528.93 MiB     mtorch                  ------------------------------ 86.61 MiB/

In [2]:
# Cell 2: Hugging Face Transformers Inference
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "RWKV/rwkv-4-169m-pile"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

prompt = "The secret to training efficient neural networks is"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate text just like a standard Transformer
outputs = model.generate(
    inputs["input_ids"],
    max_new_tokens=40,
    temperature=0.8,
    do_sample=True
)

print("\n--- Output ---")
print(tokenizer.decode(outputs[0].tolist(), skip_special_tokens=True))

Loading RWKV/rwkv-4-169m-pile...


config.json:   0%|          | 0.00/521 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  677MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [ ]:
# Cell 3: Download Native Weights and Tokenizer
import os
import urllib.request

model_file = "RWKV-4-Pile-169M.pth"
model_url = "https://huggingface.co/BlinkDL/rwkv-4-pile-169m/resolve/main/RWKV-4-Pile-169M-20220807-8023.pth"

tokenizer_file = "20B_tokenizer.json"
tokenizer_url = "https://raw.githubusercontent.com/BlinkDL/ChatRWKV/main/v2/20B_tokenizer.json"

if not os.path.exists(model_file):
    print("Downloading raw RWKV weights (approx 330MB)...")
    urllib.request.urlretrieve(model_url, model_file)

if not os.path.exists(tokenizer_file):
    print("Downloading tokenizer...")
    urllib.request.urlretrieve(tokenizer_url, tokenizer_file)
    
print("Assets ready!")

In [ ]:
# Cell 4: Native RWKV Token-by-Token Generation
import os
# These environment variables MUST be set before importing RWKV
os.environ["RWKV_JIT_ON"] = "1" # Enables PyTorch JIT for faster CPU execution

from rwkv.model import RWKV
from rwkv.utils import PIPELINE

# Load the model (using CPU and standard 32-bit floats for compatibility)
model = RWKV(model="RWKV-4-Pile-169M", strategy='cpu fp32')
pipeline = PIPELINE(model, "20B_tokenizer.json")

prompt = "In a shocking finding, scientists discovered"
print("Prompt:", prompt)
print("Output: ", end="")

# 1. Encode the prompt and get the initial state
tokens = pipeline.encode(prompt)
out, state = model.forward(tokens, None)

# 2. RNN Generation Loop
for i in range(40):
    # Sample the next token based on the output probabilities
    token = pipeline.sample_logits(out, temperature=1.0, top_p=0.85)
    
    # Decode and print the token immediately
    print(pipeline.decode([token]), end="", flush=True)
    
    # Feed ONLY the single new token and the previous state back into the model
    out, state = model.forward([token], state)